# Module 2 — Cleaning the World Bank Raw Data

**Goal:** turn the 34 raw indicator CSVs in `data/raw/world_bank/` into two clean,
analysis-ready files:

- `data/cleaned/world_bank_long.csv` — one row per (country, year, indicator)
- `data/cleaned/world_bank_wide.csv` — one row per (country, year), one column per indicator,
  using the friendly `variable_name` from `config/indicators.csv` instead of raw codes

This notebook is deliberately generic about filenames — it reads whatever `.csv` files exist
in the raw folder rather than hardcoding indicator codes. That matters because two of your
codes changed mid-project (the WGI renaming, the CO2 replacement) — a hardcoded file list
would have silently broken today. Never hardcode what you can discover from the filesystem.

In [24]:
import os
from pathlib import Path

# Walk up until we find the project root (marked by config/ and scripts/ both existing)
while not (Path("config").exists() and Path("scripts").exists()):
    os.chdir("..")
    if Path.cwd() == Path.cwd().parent:  # hit filesystem root, stop
        raise RuntimeError("Could not locate project root")

print("Working directory set to:", os.getcwd())

Working directory set to: c:\Users\AJAY\Desktop\startup_implementation


In [25]:
import pandas as pd
from pathlib import Path
import json

RAW_DIR = Path("data/raw/world_bank")
CLEANED_DIR = Path("data/cleaned")
METADATA_DIR = Path("data/metadata")
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

START_YEAR, END_YEAR = 2005, 2025

files = sorted(RAW_DIR.glob("*.csv"))
print(f"{len(files)} raw indicator files found")


35 raw indicator files found


## 1. Load and validate schema

Every file should have the same 5 columns. Rather than assume that, check it — if Module 1
ever pulls a malformed file (a different World Bank source with a different response shape),
we want a loud error here, not a silent bad merge three steps later.

In [26]:
EXPECTED_COLS = {"country_code", "country_name", "year", "indicator_code", "value"}

frames = []
schema_errors = []

for f in files:
    df = pd.read_csv(f)
    if set(df.columns) != EXPECTED_COLS:
        schema_errors.append((f.name, list(df.columns)))
        continue
    frames.append(df)

if schema_errors:
    print("Schema mismatches (excluded from merge):")
    for name, cols in schema_errors:
        print(" -", name, cols)
else:
    print("All files match the expected schema.")

print(f"{len(frames)} files loaded successfully")


All files match the expected schema.
35 files loaded successfully


## 2. Remove World Bank aggregates

As covered in the data-sources audit notebook: World Bank mixes real countries with regional
and income-group aggregates (`WLD`, `EAS`, `OECD`, ...) using the same 3-letter code column.
The only reliable way to separate them is the World Bank country-metadata endpoint, which
tags each entity's `region` as `"Aggregates"` or a real region.

This needs live internet access, so it's wrapped in a fallback: if it can't reach the API
(as in this sandbox), it uses a cached local copy if one exists from a previous run, and
otherwise stops with a clear instruction rather than silently keeping aggregates in the data.

In [27]:
import requests

def fetch_wb_country_metadata(cache_path="data/raw/world_bank_country_metadata.csv"):
    cache = Path(cache_path)
    cache.parent.mkdir(parents=True, exist_ok=True)
    try:
        url = "https://api.worldbank.org/v2/country?format=json&per_page=400"
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        _, records = r.json()
        meta = pd.DataFrame(records)
        meta["region_value"] = meta["region"].apply(lambda x: x.get("value"))
        meta["is_aggregate"] = meta["region_value"].eq("Aggregates")
        meta = meta[["id", "name", "region_value", "is_aggregate"]]
        meta.to_csv(cache, index=False)
        print(f"Fetched live country metadata: {len(meta)} entities "
              f"({meta.is_aggregate.sum()} aggregates, {(~meta.is_aggregate).sum()} countries)")
        return meta
    except requests.exceptions.RequestException as e:
        if cache.exists():
            print(f"Live fetch failed ({e.__class__.__name__}) — using cached copy from {cache}")
            return pd.read_csv(cache)
        raise RuntimeError(
            "Could not reach api.worldbank.org and no cached country metadata exists. "
            "Run this notebook with internet access at least once to build the cache."
        ) from e

wb_meta = fetch_wb_country_metadata()
aggregate_codes = set(wb_meta.loc[wb_meta.is_aggregate, "id"])


Fetched live country metadata: 295 entities (78 aggregates, 217 countries)


In [28]:
long_df = pd.concat(frames, ignore_index=True)
before = len(long_df)

long_df = long_df[long_df["country_code"].notna() & (long_df["country_code"].str.strip() != "")]
long_df = long_df[~long_df["country_code"].isin(aggregate_codes)]
after_agg = len(long_df)

long_df = long_df[(long_df["year"] >= START_YEAR) & (long_df["year"] <= END_YEAR)]
after_year = len(long_df)

long_df = long_df.drop_duplicates(subset=["country_code", "year", "indicator_code"])
after_dedup = len(long_df)

print(f"Rows: {before} -> {after_agg} after removing aggregates "
      f"-> {after_year} after year filter -> {after_dedup} after dedup")
print(f"Countries remaining: {long_df.country_code.nunique()}")


Rows: 154817 -> 129207 after removing aggregates -> 129207 after year filter -> 129207 after dedup
Countries remaining: 217


## 3. Save the long format

This is the tidy, one-fact-per-row version — useful for any tool that expects long/tidy data
(ggplot-style plotting, some statsmodels workflows, groupby aggregations).

In [29]:
long_df.to_csv(CLEANED_DIR / "world_bank_long.csv", index=False)
print(f"Saved {CLEANED_DIR / 'world_bank_long.csv'} — {len(long_df)} rows")
long_df.head()

Saved data\cleaned\world_bank_long.csv — 129207 rows


,country_code,country_name,year,indicator_code,value
987,AFG,Afghanistan,2021,BX.KLT.DINV.CD.WD,2.060098e+07
988,AFG,Afghanistan,2020,BX.KLT.DINV.CD.WD,1.297015e+07
989,AFG,Afghanistan,2019,BX.KLT.DINV.CD.WD,2.340455e+07
990,AFG,Afghanistan,2018,BX.KLT.DINV.CD.WD,1.194351e+08
991,AFG,Afghanistan,2017,BX.KLT.DINV.CD.WD,5.153390e+07


## 4. Pivot to wide format, with friendly column names

One row per (country, year); one column per indicator. Column names come from
`variable_name` in `config/indicators.csv` (e.g. `NY.GDP.MKTP.CD` -> `gdp_usd`) rather than
raw codes, so downstream stats/ML code stays readable.

Any indicator code present in the raw data but missing from `indicators.csv` (or vice versa)
gets flagged rather than silently dropped or left with a cryptic column name.

In [30]:
indicators_map = pd.read_csv("config/indicators.csv")
code_to_name = dict(zip(indicators_map["indicator_code"], indicators_map["variable_name"]))

present_codes = set(long_df["indicator_code"].unique())
mapped_codes = set(code_to_name.keys())

unmapped_in_data = present_codes - mapped_codes
unmapped_in_config = mapped_codes - present_codes

if unmapped_in_data:
    print("WARNING - indicator codes in raw data with no entry in indicators.csv:", unmapped_in_data)
if unmapped_in_config:
    print("NOTE - indicators.csv codes not found in any raw file (not downloaded / renamed?):", unmapped_in_config)


In [31]:
wide_df = long_df.pivot_table(
    index=["country_code", "year"],
    columns="indicator_code",
    values="value",
    aggfunc="first"
).reset_index()

wide_df = wide_df.merge(
    wb_meta[["id", "name"]].rename(columns={"id": "country_code", "name": "country_name"}),
    on="country_code",
    how="left"
)
wide_df = wide_df.rename(columns=code_to_name)
wide_df.columns.name = None

wide_df.to_csv(CLEANED_DIR / "world_bank_wide.csv", index=False)
print(f"Saved {CLEANED_DIR / 'world_bank_wide.csv'} — shape {wide_df.shape}")
wide_df.head()


Saved data\cleaned\world_bank_wide.csv — shape (4557, 38)


,country_code,year,fdi_net_inflows,renewable_energy,co2_per_capita,population_density,broad_money,inflation,domestic_credit,rd_expenditure,...,gdp_per_capita,tertiary_enrollment,education_expenditure,labor_force_participation,unemployment_rate,population_growth,population,urban_population_percent,hightech_exports,country_name
0,ABW,2005,-2.077980e+08,0.2,3.946331,542.416667,57.446790,3.397787,54.060701,NaN,...,24171.837133,36.204231,4.62322,NaN,NaN,2.590757,97635.0,64.958023,NaN,Aruba
1,ABW,2006,2.203161e+08,0.2,4.267391,552.250000,56.196562,3.608024,53.884120,NaN,...,24845.658484,34.226871,NaN,NaN,NaN,1.796638,99405.0,64.811270,NaN,Aruba
2,ABW,2007,-4.710056e+08,0.2,4.574139,556.388889,53.544999,5.392568,51.727256,NaN,...,26736.308944,34.896809,4.70255,NaN,NaN,0.746665,100150.0,64.653266,NaN,Aruba
3,ABW,2008,1.888268e+07,0.2,4.503701,560.650000,58.783489,8.955987,50.658458,NaN,...,28171.909401,35.097061,4.82730,NaN,NaN,0.762933,100917.0,64.488881,NaN,Aruba
4,ABW,2009,-1.061453e+07,0.3,4.855124,564.466667,69.203660,-2.135429,55.893041,NaN,...,25134.771230,34.430859,5.79740,NaN,NaN,0.678451,101604.0,64.322985,4.012599,Aruba


## 5. Cleaning report and missing-values summary

For the methodology section of your report: a machine-readable record of what this step
did, plus a per-variable missingness table so you know which indicators need imputation
decisions before modeling (and which are too sparse to keep).

In [32]:
cleaning_report = {
    "run_at": pd.Timestamp.now().isoformat(),
    "raw_files_found": len(files),
    "raw_files_loaded": len(frames),
    "schema_errors": schema_errors,
    "study_period": f"{START_YEAR}-{END_YEAR}",
    "rows_before_cleaning": before,
    "rows_after_removing_aggregates": after_agg,
    "rows_after_year_filter": after_year,
    "rows_after_dedup": after_dedup,
    "countries_remaining": int(long_df.country_code.nunique()),
    "aggregate_codes_removed": sorted(aggregate_codes & set(long_df["country_code"].unique().tolist()) | (aggregate_codes)),
    "unmapped_indicator_codes_in_data": sorted(unmapped_in_data),
    "indicators_missing_from_raw_data": sorted(unmapped_in_config),
    "wide_shape": list(wide_df.shape),
}

with open(METADATA_DIR / "cleaning_report.json", "w") as f:
    json.dump(cleaning_report, f, indent=2, default=str)

print(f"Saved {METADATA_DIR / 'cleaning_report.json'}")


Saved data\metadata\cleaning_report.json


In [33]:
missing = wide_df.drop(columns=["country_code", "country_name", "year"]).isna().mean().sort_values(ascending=False)
missing_df = missing.reset_index()
missing_df.columns = ["variable", "pct_missing"]
missing_df.to_csv(METADATA_DIR / "missing_values.csv", index=False)

print(f"Saved {METADATA_DIR / 'missing_values.csv'}")
missing_df.head(15)


Saved data\metadata\missing_values.csv


,variable,pct_missing
0,rd_expenditure,0.610270
1,tertiary_enrollment,0.427913
2,hightech_exports,0.415185
3,education_expenditure,0.387755
4,new_business_density,0.361203
5,new_businesses_registered,0.361203
6,broad_money,0.320386
7,secure_servers,0.299320
8,domestic_credit,0.254334
9,gross_capital_formation,0.211323


## Next: Module 3

With `world_bank_wide.csv` in place, Module 3 merges in the external sources from the audit
notebook — UNESCO (researchers per capita), WIPO (patents/trademarks, manual export), plus
whatever supplementary startup-ecosystem columns you decide to add (GEM TEA rate,
StartupBlink/GSER rank) — joined on `country_code` + `year`.

In [34]:
import pandas as pd
w = pd.read_csv("data/cleaned/world_bank_wide.csv")

print("Distinct countries:", w["country_code"].nunique())
print("Distinct (country, year) pairs:", w[["country_code","year"]].drop_duplicates().shape[0])
print("Total rows:", w.shape[0])

Distinct countries: 217
Distinct (country, year) pairs: 4557
Total rows: 4557
